# AeroPulse PM2.5 — Phase 2: Event-Aware Dataset + Feature Engineering

Transform the persisted Phase-1 canonical dataset and the Phase-7 driver layers into a
leakage-safe, event-aware, multi-horizon modelling dataset.

**Dataset version 3.0.0.** This is a rewrite of the v2 feature set, not an increment.

## Objective

Consume **only persisted data** — the Phase-1 canonical dataset plus the three driver
layers written by `05_pm25_driver_acquisition.ipynb`. No source is re-fetched here.

### What changed from v2, and why

v2 produced 96 features and a model with a **−103 µg/m³ bias on extreme air**. Three of
its properties caused information to be missing or wrong, and all three are fixed here.

**1. Windows counted rows, not hours.** v2 used `groupby().rolling(24)`, which takes the
last 24 *observations*. Station coverage is 86% and 3.3% of consecutive observations are
more than an hour apart — with a maximum gap of 7,320 hours. So `pm25_roll_mean_24h`
sometimes averaged over 305 days. Every feature here is computed on a **complete hourly
grid** (149 stations × 13,620 hours), so a 24-hour window is 24 hours. Padding rows are
dropped before the dataset is written.

**2. Fire proximity was not fire attribution.** A fire 40 km upwind and the same fire
40 km downwind produced identical features, so the model could not distinguish smoke
that will arrive from smoke that will not. The v2 manifest recorded this as a known gap.
It is closed here: bearings are computed per detection and compared against the wind.

**3. There was no atmospheric chemistry, terrain, or festival calendar.** AOD is the
standard input for hyper-local PM2.5 estimation, and Diwali moves through the Gregorian
calendar so `sin_doy`/`cos_doy` cannot represent it.

### The one rule everything obeys

> A feature at time `t` may use information available at or before `t`. Nothing else.

Every window below ends at `t` or earlier, every target uses a negative shift, and the
audit section at the end asserts it rather than assuming it.

### Inputs and output

```text
in  data/pm25/processed/base/base_dataset.parquet        target + station identity
    data/pm25/processed/base/weather_hourly.parquet      complete hourly weather grid
    data/pm25/processed/base/fire_detections.parquet     1.69M raw FIRMS detections
    data/pm25/processed/base/satellite_hourly.parquet    CAMS composition
    data/pm25/processed/base/static_geography.parquet    terrain
    data/pm25/processed/base/calendar_events.parquet     festivals / burning windows
out data/pm25/processed/event_aware/pm25_event_aware_features.parquet
    data/pm25/processed/event_aware/dataset_manifest.json
```

In [1]:
# ============================================================================
# CONFIGURATION
# ============================================================================
from pathlib import Path
import os, json, math, warnings, hashlib, time
import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view

from dotenv import find_dotenv, load_dotenv

_dotenv_path = find_dotenv(usecwd=True)
if _dotenv_path:
    load_dotenv(_dotenv_path, override=True)
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(os.getenv("AEROPULSE_PROJECT_ROOT", ".")).resolve()
DATA_ROOT = PROJECT_ROOT / "data" / "pm25"
PROCESSED_DIR = DATA_ROOT / "processed"
BASE_DIR = PROCESSED_DIR / "base"
EVENT_DIR = PROCESSED_DIR / "event_aware"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "pm25"
EVENT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "dataset_version": "3.0.0",

    # Temporal memory. 168h (one week) is the longest: multi-day winter episodes
    # build slowly and a 48h horizon truncates that history.
    "lags": [1, 2, 3, 6, 12, 24, 48, 72, 168],
    "roll_mean_windows": [3, 6, 12, 24, 48, 72, 168],
    "roll_std_windows": [6, 12, 24, 48],
    "delta_windows": [1, 3, 6, 12, 24],
    "slope_windows": [3, 6, 12, 24],

    # Fire geometry. 5km resolves a local burn, 150km covers the Punjab ->
    # Delhi transport distance.
    "fire_radii_km": [5, 10, 20, 50, 100, 150],
    "fire_upwind_radii_km": [10, 20, 50, 100],
    "fire_window_hours": 24,
    "fire_decay_halflives_h": [6, 12, 24, 48],
    "fire_decay_radius_km": 100,

    # Direction is discretised into sectors so that upwind selection is a table
    # lookup rather than a per-row angular test. 16 sectors = 22.5 deg each.
    "n_sectors": 16,
    "upwind_max_deg": 45.0,      # |bearing - wind_from| <= 45 => upwind
    "downwind_min_deg": 135.0,

    "regional_radii_km": [25, 50, 100],

    "horizons": [1, 3, 6, 12, 24, 48],
    "peak_windows": [6, 12, 24, 48],
    "event_threshold": float(os.getenv("PM25_EVENT_THRESHOLD", "60")),
    "regime_bins": [-np.inf, 30, 60, 90, 150, np.inf],
    "regime_labels": ["normal", "moderate", "high", "very_high", "extreme"],
}

print("Project :", PROJECT_ROOT)
for k in ["dataset_version", "lags", "fire_radii_km", "regional_radii_km", "horizons"]:
    print(f"  {k}: {CONFIG[k]}")

Project : /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator
  dataset_version: 3.0.0
  lags: [1, 2, 3, 6, 12, 24, 48, 72, 168]
  fire_radii_km: [5, 10, 20, 50, 100, 150]
  regional_radii_km: [25, 50, 100]
  horizons: [1, 3, 6, 12, 24, 48]


## 1. Load Phase-1 and the driver layers

Each load asserts its contract. A Phase-1 or Phase-7 regression then fails here with a
named column instead of surfacing as an all-NaN feature three notebooks downstream.

In [2]:
# ============================================================================
# LOAD - target layer, drivers, and their contracts
# ============================================================================
BASE_FILE = BASE_DIR / "base_dataset.parquet"
WX_FILE = BASE_DIR / "weather_hourly.parquet"
FIRE_FILE = BASE_DIR / "fire_detections.parquet"
SAT_FILE = BASE_DIR / "satellite_hourly.parquet"
GEO_FILE = BASE_DIR / "static_geography.parquet"
CAL_FILE = BASE_DIR / "calendar_events.parquet"

for f in [BASE_FILE, WX_FILE, FIRE_FILE]:
    if not f.exists():
        raise FileNotFoundError(f"{f} not found. Run Phase 1 first.")
for f in [SAT_FILE, GEO_FILE, CAL_FILE]:
    if not f.exists():
        raise FileNotFoundError(
            f"{f} not found. Run 05_pm25_driver_acquisition.ipynb first.")

OBS_COLS = ["timestamp_utc", "location_id", "sensor_id", "station_name",
            "lat", "lon", "pm25", "source", "source_quality"]
obs = pd.read_parquet(BASE_FILE, columns=OBS_COLS)
obs["timestamp_utc"] = pd.to_datetime(obs["timestamp_utc"], utc=True)
obs["lat"] = obs["lat"].round(4)
obs["lon"] = obs["lon"].round(4)

WEATHER_VARS = ["temperature_2m", "relative_humidity_2m", "surface_pressure",
                "precipitation", "wind_speed_10m", "wind_direction_10m",
                "boundary_layer_height", "wind_u", "wind_v"]
weather = pd.read_parquet(WX_FILE)
weather["timestamp_utc"] = pd.to_datetime(weather["timestamp_utc"], utc=True)
weather["lat"] = weather["lat"].round(4)
weather["lon"] = weather["lon"].round(4)

fire = pd.read_parquet(FIRE_FILE)
fire["timestamp_utc"] = pd.to_datetime(fire["timestamp_utc"], utc=True)

satellite = pd.read_parquet(SAT_FILE)
satellite["timestamp_utc"] = pd.to_datetime(satellite["timestamp_utc"], utc=True)
satellite["lat"] = satellite["lat"].round(4)
satellite["lon"] = satellite["lon"].round(4)
CAMS_VARS = [c for c in satellite.columns if c.startswith("cams_")]

terrain = pd.read_parquet(GEO_FILE)
terrain["lat"] = terrain["lat"].round(4)
terrain["lon"] = terrain["lon"].round(4)
calendar = pd.read_parquet(CAL_FILE)

missing = [c for c in WEATHER_VARS if c not in weather.columns]
if missing:
    raise ValueError(f"weather_hourly is missing {missing}")

print(f"observations : {len(obs):>10,} rows, {obs.location_id.nunique()} stations")
print(f"               {obs.timestamp_utc.min()} -> {obs.timestamp_utc.max()}")
print(f"weather      : {len(weather):>10,} rows")
print(f"fire         : {len(fire):>10,} detections")
print(f"satellite    : {len(satellite):>10,} rows, vars {CAMS_VARS}")
print(f"terrain      : {len(terrain):>10,} stations")
print(f"calendar     : {len(calendar):>10,} dates")

observations :  1,705,252 rows, 149 stations
               2025-02-18 20:30:00+00:00 -> 2026-09-09 07:30:00+00:00
weather      :  2,034,744 rows
fire         :  1,692,885 detections
satellite    :  2,034,744 rows, vars ['cams_pm25', 'cams_pm10', 'cams_aod', 'cams_no2', 'cams_co', 'cams_so2', 'cams_dust', 'cams_o3']
terrain      :        149 stations
calendar     :        589 dates


## 2. Build the complete station-hour grid

This is the structural change that makes every window below mean what it says.

The v2 pipeline computed rolling windows over *rows*. Station coverage is 86%, so a
"24-hour" mean was frequently a mean over more than 24 hours — and for the station with
a 7,320-hour gap, over most of a year. Reindexing onto a dense hourly grid makes a row
offset and an hour offset the same thing, which is also what allows the whole feature
set to be computed as matrix operations rather than 2 million grouped windows.

Gap hours enter the grid with `pm25 = NaN`. They are **not interpolated** — a filled
target would be a fabricated observation. They exist so that windows are correctly
spaced, and they are dropped before the dataset is written.

In [3]:
# ============================================================================
# STATION-HOUR GRID - the spine everything else is computed on
# ============================================================================
# Shape is (S stations, T hours). Every feature below is an (S, T) matrix, and
# the frame is assembled at the end by ravelling in C order (station-major),
# which is why STATIONS and GRID_HOURS must not be reordered after this cell.
# ============================================================================
stations = (obs.groupby("location_id", as_index=False)
               .agg(sensor_id=("sensor_id", "first"),
                    station_name=("station_name", "first"),
                    lat=("lat", "first"), lon=("lon", "first"),
                    source=("source", "first"),
                    source_quality=("source_quality", "first"),
                    n_obs=("pm25", "size"))
               .sort_values("location_id")
               .reset_index(drop=True))

GRID_HOURS = pd.date_range(obs.timestamp_utc.min(), obs.timestamp_utc.max(), freq="h")
S, T = len(stations), len(GRID_HOURS)
sid_pos = {sid: i for i, sid in enumerate(stations.location_id)}
hour_pos = pd.Series(np.arange(T), index=GRID_HOURS)

# Scatter the observed PM2.5 into the grid. Duplicate station-hours would
# silently overwrite here, so assert their absence rather than trusting Phase 1.
dupes = obs.duplicated(subset=["location_id", "timestamp_utc"]).sum()
if dupes:
    raise ValueError(f"{dupes:,} duplicate station-hours in Phase 1 output")

PM25 = np.full((S, T), np.nan, dtype="float32")
row = obs["location_id"].map(sid_pos).to_numpy()
col = obs["timestamp_utc"].map(hour_pos).to_numpy()
PM25[row, col] = obs["pm25"].to_numpy(dtype="float32")

observed = ~np.isnan(PM25)
print(f"Grid          : {S} stations x {T:,} hours = {S*T:,} cells")
print(f"Observed      : {observed.sum():,} ({observed.mean():.1%} of the grid)")
print(f"Padding rows  : {(~observed).sum():,} (dropped before writing)")
print(f"Span          : {GRID_HOURS[0]} -> {GRID_HOURS[-1]}")
print(f"Grid spacing  : {(GRID_HOURS[1]-GRID_HOURS[0])}")
print("\nCoverage per station: median {:.1%}, worst {:.1%}".format(
    np.median(observed.mean(axis=1)), observed.mean(axis=1).min()))

Grid          : 149 stations x 13,620 hours = 2,029,380 cells
Observed      : 1,705,252 (84.0% of the grid)
Padding rows  : 324,128 (dropped before writing)
Span          : 2025-02-18 20:30:00+00:00 -> 2026-09-09 07:30:00+00:00
Grid spacing  : 0 days 01:00:00

Coverage per station: median 86.3%, worst 44.6%


### A causal rolling toolkit

Every window statistic in this notebook goes through these five helpers, so causality is
enforced in one place instead of being re-argued at each call site.

The `shift` argument is the key: `shift=1` means the window **ends one hour before `t`**,
so the value at `t` cannot enter its own feature. Statistics that deliberately compare
the current value against its own history (z-score, percentile) use `shift=1` for the
history and then bring in `pm25[t]` explicitly — `pm25[t]` is known at prediction time,
which is why Phase 5 includes it as a feature.

In [4]:
# ============================================================================
# CAUSAL ROLLING TOOLKIT  (all operate on (S, T) arrays along the time axis)
# ============================================================================
def _cumsum0(x):
    """Cumulative sum with a leading zero column, so a window sum is C[hi]-C[lo]."""
    return np.concatenate(
        [np.zeros((x.shape[0], 1), dtype="float64"), np.cumsum(x, axis=1, dtype="float64")],
        axis=1)


def _bounds(w, shift):
    """Half-open [lo, hi) column bounds of the window of width w ending at t-shift."""
    hi = np.clip(np.arange(T) - shift + 1, 0, T)
    lo = np.clip(hi - w, 0, T)
    return lo, hi


def roll_mean_std(a, w, shift=1, min_periods=None):
    """Trailing mean and std over w hours ending `shift` hours before t."""
    min_periods = min_periods or max(2, w // 4)
    valid = ~np.isnan(a)
    x = np.where(valid, a, 0.0)
    cx, cx2, cn = _cumsum0(x), _cumsum0(x * x), _cumsum0(valid.astype("float64"))
    lo, hi = _bounds(w, shift)
    n = cn[:, hi] - cn[:, lo]
    s = cx[:, hi] - cx[:, lo]
    s2 = cx2[:, hi] - cx2[:, lo]
    ok = n >= min_periods
    mean = np.where(ok, s / np.where(n == 0, 1, n), np.nan)
    var = np.where(ok, s2 / np.where(n == 0, 1, n) - mean ** 2, np.nan)
    return mean.astype("float32"), np.sqrt(np.clip(var, 0, None)).astype("float32")


def roll_sum(a, w, shift=1):
    """Trailing sum, treating NaN as absent (not as zero contribution to count)."""
    x = np.where(np.isnan(a), 0.0, a)
    c = _cumsum0(x)
    lo, hi = _bounds(w, shift)
    return (c[:, hi] - c[:, lo]).astype("float32")


def roll_windows(a, w, shift=1):
    """(S, T, w) view of the trailing window. Materialised, so chunk large w."""
    pad = np.full((a.shape[0], w + shift - 1), np.nan, dtype=a.dtype)
    padded = np.concatenate([pad, a], axis=1)
    return sliding_window_view(padded, w, axis=1)[:, :T, :]


def roll_slope(a, w, shift=1, min_periods=None):
    """Ordinary-least-squares slope (per hour) over the trailing window.

    Not a rescaled delta. A delta uses two endpoints, so one noisy reading moves
    it; the OLS slope uses every point in the window, which is what makes it a
    different signal rather than a monotone transform trees would ignore.

    slope = sum((x-xbar)(y-ybar)) / sum((x-xbar)^2), x = 0..w-1 within the window.
    The denominator is the closed form w(w^2-1)/12. With NaNs present xbar is
    approximated by the full-window value, which biases the slope slightly on
    gappy windows - acceptable, and min_periods bounds how gappy that can be.
    """
    min_periods = min_periods or max(3, w // 3)
    valid = ~np.isnan(a)
    y = np.where(valid, a, 0.0)
    k = np.arange(T, dtype="float64")[None, :]
    cy, cky, cn = _cumsum0(y), _cumsum0(y * k), _cumsum0(valid.astype("float64"))
    lo, hi = _bounds(w, shift)
    n = cn[:, hi] - cn[:, lo]
    sy = cy[:, hi] - cy[:, lo]
    sky = cky[:, hi] - cky[:, lo]
    # True (unclipped) window start, so x stays 0..w-1 even where the window
    # runs off the start of the series - those columns simply contribute nothing.
    c = (np.arange(T) - shift + 1 - w).astype("float64")[None, :]
    num = (sky - c * sy) - ((w - 1) / 2.0) * sy
    den = w * (w * w - 1) / 12.0
    return np.where(n >= min_periods, num / den, np.nan).astype("float32")


def naive64(x):
    """tz-aware datetimes -> datetime64[ns] for arithmetic.

    pandas 3 returns an OBJECT array from .to_numpy() on a tz-aware series, and
    object arrays cannot be subtracted from datetime64. Every place this
    notebook does timestamp arithmetic goes through here.
    """
    idx = pd.DatetimeIndex(x)
    return idx.tz_convert("UTC").tz_localize(None).to_numpy("datetime64[ns]")


def shift_time(a, k):
    """Lag (k>0) or lead (k<0) along the time axis, NaN-filled at the edges."""
    out = np.full_like(a, np.nan)
    if k > 0:
        out[:, k:] = a[:, :-k]
    elif k < 0:
        out[:, :k] = a[:, -k:]
    else:
        out[:] = a
    return out


print("Rolling toolkit ready: roll_mean_std, roll_sum, roll_windows, roll_slope, "
      "shift_time, naive64")

Rolling toolkit ready: roll_mean_std, roll_sum, roll_windows, roll_slope, shift_time, naive64


## 3. Temporal PM2.5 features

Level, memory, dynamics and anomaly — the four things a forecaster needs from the target's
own history.

Dynamics matter most for the problem this project has: a model that sees only the level
learns to predict the level, which is what produces a hedged forecast during an episode.
A model that sees the *rate of rise* can predict where the level is going.

In [5]:
# ============================================================================
# AUTOREGRESSIVE FEATURES - level, memory, dynamics, anomaly
# ============================================================================
feat = {}          # name -> (S, T) float32 matrix
t0 = time.time()

feat["pm25"] = PM25                                    # known at t; not a leak
for lag in CONFIG["lags"]:
    feat[f"pm25_lag_{lag}h"] = shift_time(PM25, lag)

for w in CONFIG["roll_mean_windows"]:
    m, s = roll_mean_std(PM25, w, shift=1)
    feat[f"pm25_roll_mean_{w}h"] = m
    if w in CONFIG["roll_std_windows"]:
        feat[f"pm25_roll_std_{w}h"] = s

# Shape of the last day: a 24h mean cannot distinguish a flat day from one that
# spiked for three hours, and it is the spike that matters for alerting.
# Chunked over stations - the materialised (S, T, 24) window is 195 MB and the
# quantile routines allocate several copies of it.
shape_names = ["pm25_roll_min_24h", "pm25_roll_max_24h", "pm25_roll_median_24h",
               "pm25_roll_p90_24h", "pm25_roll_p95_24h"]
for n in shape_names:
    feat[n] = np.empty((S, T), dtype="float32")
for lo_i in range(0, S, 25):
    hi_i = min(lo_i + 25, S)
    w24 = roll_windows(PM25[lo_i:hi_i], 24, shift=1)
    feat["pm25_roll_min_24h"][lo_i:hi_i] = np.nanmin(w24, axis=2)
    feat["pm25_roll_max_24h"][lo_i:hi_i] = np.nanmax(w24, axis=2)
    feat["pm25_roll_median_24h"][lo_i:hi_i] = np.nanmedian(w24, axis=2)
    feat["pm25_roll_p90_24h"][lo_i:hi_i] = np.nanpercentile(w24, 90, axis=2)
    feat["pm25_roll_p95_24h"][lo_i:hi_i] = np.nanpercentile(w24, 95, axis=2)
    del w24

for w in CONFIG["delta_windows"]:
    feat[f"pm25_delta_{w}h"] = (PM25 - feat[f"pm25_lag_{w}h"]).astype("float32")
feat["pm25_pct_change_1h"] = (feat["pm25_delta_1h"] /
                              np.where(feat["pm25_lag_1h"] == 0, np.nan,
                                       feat["pm25_lag_1h"])).astype("float32")
feat["pm25_pct_change_6h"] = (feat["pm25_delta_6h"] /
                              np.where(feat["pm25_lag_6h"] == 0, np.nan,
                                       feat["pm25_lag_6h"])).astype("float32")

for w in CONFIG["slope_windows"]:
    feat[f"pm25_slope_{w}h"] = roll_slope(PM25, w, shift=0)

# Is the rise itself accelerating? This separates an episode that is building
# from one that has plateaued - the two look identical to a level feature.
feat["pm25_acceleration_3h"] = (feat["pm25_delta_3h"] -
                                shift_time(feat["pm25_delta_3h"], 3)).astype("float32")

# Anomaly: how unusual is NOW relative to THIS station's recent history? An
# absolute 80 ug/m3 is routine in Delhi and extraordinary in Chennai, and a
# model trained across both needs the relative signal as well as the absolute.
for w in (24, 72):
    m, s = roll_mean_std(PM25, w, shift=1)
    feat[f"pm25_zscore_{w}h"] = ((PM25 - m) / np.where(s < 1e-3, np.nan, s)).astype("float32")

# Percentile within the trailing week. Chunked over stations because the
# materialised (S, T, 168) window would be 1.4 GB in one piece.
pct = np.full((S, T), np.nan, dtype="float32")
for i in range(S):
    w168 = roll_windows(PM25[i:i + 1], 168, shift=1)[0]      # (T, 168)
    n = np.sum(~np.isnan(w168), axis=1)
    below = np.sum(w168 < PM25[i][:, None], axis=1)
    pct[i] = np.where(n >= 24, below / np.where(n == 0, 1, n), np.nan)
feat["pm25_percentile_7d"] = pct
del pct

print(f"Temporal features: {len(feat)} in {time.time()-t0:.0f}s")
print("\nLeakage probe - a shifted window must not reproduce the current value:")
same = np.nanmean((np.round(feat['pm25_roll_mean_3h'], 4) == np.round(PM25, 4)))
print(f"  rows where roll_mean_3h == pm25: {same:.2%} "
      f"(flat stretches only; above ~10% would mean the shift was lost)")
print(f"  pm25_lag_168h null rate       : {np.isnan(feat['pm25_lag_168h']).mean():.1%}")
print(f"  pm25_percentile_7d null rate  : {np.isnan(feat['pm25_percentile_7d']).mean():.1%}")

Temporal features: 41 in 67s

Leakage probe - a shifted window must not reproduce the current value:
  rows where roll_mean_3h == pm25: 1.51% (flat stretches only; above ~10% would mean the shift was lost)
  pm25_lag_168h null rate       : 17.0%
  pm25_percentile_7d null rate  : 2.8%


## 4. Weather on the grid, and dispersion physics

PM2.5 concentration is emission divided by the volume of air it mixes into. The weather
block describes that denominator.

Weather is placed on the grid by gathering rather than joining: the weather layer is
already a dense hourly grid on the same 149 coordinates, so the value for grid hour
`t` is the row at `floor(t)`. That is the same point-in-time-safe choice `merge_asof(direction="backward")`
makes in Phase 1 — the most recent weather hour at or before `t`, never a future one —
but it is exact and avoids a 2-million-row asof join.

**Production note.** These are weather *observations* at `t`, used to forecast `t+h`. A
deployed +24 h forecast would instead use a weather *forecast* valid at `t+24`. That is
recorded as an open item (roadmap §30); it is not leakage, but it does mean these
features are weaker at long horizons than a production system's would be.

In [6]:
# ============================================================================
# WEATHER ON THE GRID + DISPERSION PHYSICS
# ============================================================================
t0 = time.time()
COORD_INDEX = pd.MultiIndex.from_arrays([stations.lat.to_numpy(), stations.lon.to_numpy()])


def gather_hourly(layer, value_cols, label):
    """Place a dense (coordinate x hour) layer onto the station-hour grid.

    Returns {col: (S, T)}. The grid sits at HH:30 and these layers at HH:00, so
    the value for grid hour t is the row at floor(t) - the most recent hour AT
    OR BEFORE t. That is the same direction merge_asof(backward) enforces in
    Phase 1, and it is the only direction that cannot pull a future value into
    a feature. Doing it by index rather than by asof join keeps a 2-million-row
    merge out of the hot path.
    """
    src_hours = pd.DatetimeIndex(np.sort(layer.timestamp_utc.unique()))
    src_pos = pd.Series(np.arange(len(src_hours)), index=src_hours)

    take = src_pos.reindex(GRID_HOURS.floor("h")).to_numpy()
    if np.isnan(take).any():
        raise ValueError(f"{label} layer does not cover every grid hour "
                         f"({int(np.isnan(take).sum())} missing)")
    take = take.astype(int)

    rows = COORD_INDEX.get_indexer(
        pd.MultiIndex.from_arrays([layer["lat"].to_numpy(), layer["lon"].to_numpy()]))
    if (rows < 0).any():
        raise ValueError(f"{label} layer has coordinates absent from the station list")
    cols = src_pos.reindex(layer["timestamp_utc"]).to_numpy().astype(int)

    out = {}
    for col in value_cols:
        buf = np.full((S, len(src_hours)), np.nan, dtype="float32")
        buf[rows, cols] = layer[col].to_numpy(dtype="float32")
        out[col] = buf[:, take]
    return out


WX = gather_hourly(weather, WEATHER_VARS, "weather")
for name, mat in WX.items():
    feat[name] = mat

# Rain accumulation. Washout is cumulative - a millimetre now matters far less
# than 40 mm over the past two days, and the monsoon regime is defined by the
# latter.
for w in [1, 3, 6, 12, 24, 48]:
    feat[f"rain_{w}h"] = roll_sum(WX["precipitation"], w, shift=0)

# Tendencies. A falling boundary layer is the mechanism of an evening build-up;
# its level alone does not say whether it is collapsing or lifting.
for src, name in [("temperature_2m", "delta_temperature"),
                  ("surface_pressure", "delta_pressure"),
                  ("boundary_layer_height", "delta_pbl"),
                  ("wind_speed_10m", "delta_wind")]:
    feat[f"{name}_3h"] = (WX[src] - shift_time(WX[src], 3)).astype("float32")
    feat[f"{name}_24h"] = (WX[src] - shift_time(WX[src], 24)).astype("float32")

# Ventilation coefficient = wind speed x mixing height: the volume flux of air
# available to dilute into, and the single best scalar summary of dispersion.
feat["ventilation_coefficient"] = (WX["wind_speed_10m"] *
                                   WX["boundary_layer_height"]).astype("float32")
feat["low_wind_flag"] = (WX["wind_speed_10m"] < 2.0).astype("float32")
feat["low_pbl_flag"] = (WX["boundary_layer_height"] < 500.0).astype("float32")
# Stagnation is the compound condition, and it is the compound that produces
# episodes: light wind under a shallow inversion with no rain to scavenge.
feat["stagnation_flag"] = ((WX["wind_speed_10m"] < 2.0) &
                           (WX["boundary_layer_height"] < 500.0)).astype("float32")
vent = feat["ventilation_coefficient"]
vmean, _ = roll_mean_std(vent, 168, shift=1)
feat["stagnation_index"] = (vmean / np.where(vent < 1e-3, np.nan, vent)).astype("float32")

print(f"Weather features added in {time.time()-t0:.0f}s")
print(f"  weather coverage on grid : {(~np.isnan(WX['temperature_2m'])).mean():.1%}")

# Physics check: the dispersion story must be visible in the raw data, or the
# join is wrong regardless of how many rows are populated.
stag = feat["stagnation_flag"] == 1
vented = (WX["wind_speed_10m"] >= 2.0) & (WX["boundary_layer_height"] >= 500.0)
print(f"  mean PM2.5 when stagnant : {np.nanmean(PM25[stag & observed]):.1f} ug/m3")
print(f"  mean PM2.5 when ventilated: {np.nanmean(PM25[vented & observed]):.1f} ug/m3")
print(f"  rows stagnant            : {stag.mean():.1%}")

Weather features added in 0s
  weather coverage on grid : 100.0%
  mean PM2.5 when stagnant : 52.5 ug/m3
  mean PM2.5 when ventilated: 28.2 ug/m3
  rows stagnant            : 4.4%


## 5. Fire: multi-radius evidence and upwind attribution

This is the largest gap v2 left open.

Radius aggregates answer "is there fire nearby". They cannot answer "will its smoke reach
this station", because a fire 40 km upwind and the same fire 40 km downwind produce
identical counts. Wind direction is the missing term, and it is available: Phase 1
persisted every detection's latitude, longitude and FRP, and the weather layer carries
`wind_direction_10m`.

For each detection the bearing from station to fire is computed and compared with the
wind direction. Meteorological convention matters here: `wind_direction_10m` is the
direction the wind blows **from**, so a fire is upwind when the bearing to it is close
to the wind direction — smoke travels from the fire toward the station.

```text
        fire            wind_from = 315 deg (NW)
          \             bearing(station -> fire) = 310 deg
           \            |difference| = 5 deg  =>  UPWIND, smoke arrives
            station
```

Bearings are bucketed into 16 sectors of 22.5° so that upwind selection becomes a table
lookup instead of a per-row angular test. The cost is discretisation: the ±45° upwind
cone is resolved to the nearest 22.5°. That is a deliberate trade for making the
computation tractable on 1.69M detections × 2M station-hours, and it is recorded in the
manifest.

In [7]:
# ============================================================================
# FIRE FEATURES - multi-radius, decayed, and direction-resolved
# ============================================================================
t0 = time.time()
R_EARTH = 6371.0088
NSEC = CONFIG["n_sectors"]
SECTOR_DEG = 360.0 / NSEC
RADII = CONFIG["fire_radii_km"]
RMAX = float(max(RADII))

fire_lat = fire["fire_lat"].to_numpy("float64")
fire_lon = fire["fire_lon"].to_numpy("float64")
fire_frp = fire["frp"].to_numpy("float32")

# Map each detection to the FIRST grid hour at or after it. Rounding DOWN would
# make a 07:12 detection visible at the 06:30 grid hour - a feature that knows
# about a fire before it burned.
offs = ((naive64(fire["timestamp_utc"]) - naive64(GRID_HOURS)[0])
        / np.timedelta64(1, "h"))
fire_bucket = np.ceil(offs).astype("int64")
in_span = (fire_bucket >= 0) & (fire_bucket < T)
print(f"Detections inside the grid span: {in_span.sum():,} / {len(fire):,}")
fire_lat, fire_lon = fire_lat[in_span], fire_lon[in_span]
fire_frp, fire_bucket = fire_frp[in_span], fire_bucket[in_span]


def haversine_km(lat1, lon1, lat2, lon2):
    dphi = np.deg2rad(lat2 - lat1)
    dl = np.deg2rad(lon2 - lon1)
    a = (np.sin(dphi / 2) ** 2 + np.cos(np.deg2rad(lat1)) *
         np.cos(np.deg2rad(lat2)) * np.sin(dl / 2) ** 2)
    return 2 * R_EARTH * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def bearing_deg(lat1, lon1, lat2, lon2):
    """Initial great-circle bearing from point 1 to point 2, degrees clockwise from N."""
    p1, p2 = np.deg2rad(lat1), np.deg2rad(lat2)
    dl = np.deg2rad(lon2 - lon1)
    y = np.sin(dl) * np.cos(p2)
    x = np.cos(p1) * np.sin(p2) - np.sin(p1) * np.cos(p2) * np.cos(dl)
    return np.rad2deg(np.arctan2(y, x)) % 360.0


# Sector-to-sector angular distance, precomputed once. sector_gap[a, b] is the
# smallest angle between the centres of sectors a and b.
_s = np.arange(NSEC)
sector_gap = np.abs(_s[:, None] - _s[None, :]) * SECTOR_DEG
sector_gap = np.minimum(sector_gap, 360.0 - sector_gap)
M_UP = (sector_gap <= CONFIG["upwind_max_deg"]).astype("float32")
M_DOWN = (sector_gap >= CONFIG["downwind_min_deg"]).astype("float32")
M_CROSS = 1.0 - M_UP - M_DOWN

# Wind sector per station-hour. NaN wind => no directional attribution is
# possible, and the result must be NaN rather than a silent default of sector 0.
wind_from = WX["wind_direction_10m"]
wind_ok = ~np.isnan(wind_from)
wind_sec = np.where(wind_ok, np.floor(np.nan_to_num(wind_from) / SECTOR_DEG), 0).astype("int64") % NSEC

fire_out = {n: np.zeros((S, T), dtype="float32") for n in
            [f"{p}_{r}km_24h" for r in RADII
             for p in ("fire_count", "fire_frp_sum", "fire_distance_weighted_frp")]}
for r in RADII:
    fire_out[f"fire_frp_max_{r}km_24h"] = np.zeros((S, T), dtype="float32")
    fire_out[f"fire_min_distance_{r}km_24h"] = np.full((S, T), np.nan, dtype="float32")
for r in CONFIG["fire_upwind_radii_km"]:
    fire_out[f"upwind_fire_count_{r}km"] = np.full((S, T), np.nan, dtype="float32")
    fire_out[f"upwind_fire_frp_{r}km"] = np.full((S, T), np.nan, dtype="float32")
for n in ["crosswind_fire_frp_100km", "downwind_fire_frp_100km",
          "upwind_fire_fraction_100km"]:
    fire_out[n] = np.full((S, T), np.nan, dtype="float32")
for hl in CONFIG["fire_decay_halflives_h"]:
    fire_out[f"fire_frp_decay_{hl}h"] = np.zeros((S, T), dtype="float32")

WIN = CONFIG["fire_window_hours"]
DECAY_R = CONFIG["fire_decay_radius_km"]

for i, st in enumerate(stations.itertuples()):
    # Cheap bounding-box prefilter before the trigonometry: a 150 km circle is
    # ~2% of India, so this discards most of the 1.69M detections immediately.
    dlat = RMAX / 111.0
    dlon = RMAX / (111.0 * max(math.cos(math.radians(st.lat)), 0.1))
    box = ((np.abs(fire_lat - st.lat) <= dlat) & (np.abs(fire_lon - st.lon) <= dlon))
    if not box.any():
        continue
    flat, flon = fire_lat[box], fire_lon[box]
    frp, bkt = fire_frp[box], fire_bucket[box]
    dist = haversine_km(st.lat, st.lon, flat, flon)
    keep = dist <= RMAX
    if not keep.any():
        continue
    dist, frp, bkt = dist[keep], frp[keep], bkt[keep]
    sec = (np.floor(bearing_deg(st.lat, st.lon, flat[keep], flon[keep]) / SECTOR_DEG)
           .astype("int64") % NSEC)

    for r in RADII:
        m = dist <= r
        if not m.any():
            continue
        # Hourly aggregates first, then one rolling pass - not a re-scan of the
        # detection list for each of the 13,620 hours.
        cnt_h = np.zeros(T, dtype="float64")
        frp_h = np.zeros(T, dtype="float64")
        dwf_h = np.zeros(T, dtype="float64")
        max_h = np.zeros(T, dtype="float64")
        min_h = np.full(T, np.inf, dtype="float64")
        np.add.at(cnt_h, bkt[m], 1.0)
        np.add.at(frp_h, bkt[m], frp[m])
        # Inverse-distance weighting: a 50 MW fire at 5 km is not the same
        # evidence as a 50 MW fire at 95 km, which a plain sum treats alike.
        np.add.at(dwf_h, bkt[m], frp[m] / (1.0 + dist[m]))
        np.maximum.at(max_h, bkt[m], frp[m])
        np.minimum.at(min_h, bkt[m], dist[m])

        one = lambda v: v[None, :]
        fire_out[f"fire_count_{r}km_24h"][i] = roll_sum(one(cnt_h), WIN, shift=0)[0]
        fire_out[f"fire_frp_sum_{r}km_24h"][i] = roll_sum(one(frp_h), WIN, shift=0)[0]
        fire_out[f"fire_distance_weighted_frp_{r}km_24h"][i] = roll_sum(one(dwf_h), WIN, shift=0)[0]
        wmax = roll_windows(one(max_h), WIN, shift=0)[0]
        fire_out[f"fire_frp_max_{r}km_24h"][i] = np.nanmax(wmax, axis=1)
        min_h[np.isinf(min_h)] = np.nan
        wmin = roll_windows(one(min_h), WIN, shift=0)[0]
        fire_out[f"fire_min_distance_{r}km_24h"][i] = np.nanmin(wmin, axis=1)

        if r in CONFIG["fire_upwind_radii_km"] or r == DECAY_R:
            # (T, NSEC) FRP and count by bearing sector, rolled over the window.
            cnt_s = np.zeros((T, NSEC), dtype="float64")
            frp_s = np.zeros((T, NSEC), dtype="float64")
            np.add.at(cnt_s, (bkt[m], sec[m]), 1.0)
            np.add.at(frp_s, (bkt[m], sec[m]), frp[m])
            cnt_sr = roll_sum(cnt_s.T, WIN, shift=0).T          # (T, NSEC)
            frp_sr = roll_sum(frp_s.T, WIN, shift=0).T
            up_mask = M_UP[wind_sec[i]]                          # (T, NSEC)
            if r in CONFIG["fire_upwind_radii_km"]:
                fire_out[f"upwind_fire_count_{r}km"][i] = np.where(
                    wind_ok[i], (cnt_sr * up_mask).sum(axis=1), np.nan)
                fire_out[f"upwind_fire_frp_{r}km"][i] = np.where(
                    wind_ok[i], (frp_sr * up_mask).sum(axis=1), np.nan)
            if r == DECAY_R:
                total = frp_sr.sum(axis=1)
                up = (frp_sr * up_mask).sum(axis=1)
                fire_out["crosswind_fire_frp_100km"][i] = np.where(
                    wind_ok[i], (frp_sr * M_CROSS[wind_sec[i]]).sum(axis=1), np.nan)
                fire_out["downwind_fire_frp_100km"][i] = np.where(
                    wind_ok[i], (frp_sr * M_DOWN[wind_sec[i]]).sum(axis=1), np.nan)
                # The fraction is the transport-aware signal: it separates "a lot
                # of fire, none of it pointed here" from "less fire, aimed at us".
                fire_out["upwind_fire_fraction_100km"][i] = np.where(
                    wind_ok[i] & (total > 0), up / np.where(total == 0, 1, total), np.nan)

        if r == DECAY_R:
            # Exponential memory of burning. Smoke does not vanish at the 24h
            # window edge, and a half-life expresses that better than a box.
            s_frp = pd.Series(frp_h)
            for hl in CONFIG["fire_decay_halflives_h"]:
                fire_out[f"fire_frp_decay_{hl}h"][i] = (
                    s_frp.ewm(halflife=hl, adjust=False).mean().to_numpy(dtype="float32"))

fire_out["fire_active_flag"] = (fire_out["fire_count_100km_24h"] > 0).astype("float32")
feat.update(fire_out)
print(f"\nFire features: {len(fire_out)} in {time.time()-t0:.0f}s")

Detections inside the grid span: 1,688,841 / 1,692,885



Fire features: 46 in 4s


In [8]:
# ============================================================================
# FIRE FEATURE VALIDATION - is the direction signal real?
# ============================================================================
# A wrong bearing convention (or a wind field 180 degrees out) would still
# produce populated columns. The test is whether upwind fire tracks PM2.5 more
# strongly than downwind fire - which is a physical prediction, not a
# statistical one.
# ============================================================================
print("Rows with a detection in the prior 24h, by radius:")
for r in RADII:
    m = feat[f"fire_count_{r}km_24h"]
    print(f"  {r:>4} km : {(m > 0).mean():6.2%} of grid, max count {m.max():.0f}, "
          f"max FRP {feat[f'fire_frp_sum_{r}km_24h'].max():,.0f} MW")

o = observed
for name in ["fire_frp_sum_100km_24h", "upwind_fire_frp_100km",
             "crosswind_fire_frp_100km", "downwind_fire_frp_100km"]:
    v = feat[name]
    ok = o & ~np.isnan(v) & (v > 0)
    none = o & ~np.isnan(v) & (v == 0)
    if ok.sum() > 1000:
        print(f"\n{name}")
        print(f"  mean PM2.5 when > 0 : {np.nanmean(PM25[ok]):6.1f} ug/m3  (n={ok.sum():,})")
        print(f"  mean PM2.5 when = 0 : {np.nanmean(PM25[none]):6.1f} ug/m3  (n={none.sum():,})")

up = feat["upwind_fire_frp_100km"]
dn = feat["downwind_fire_frp_100km"]
sel = o & ~np.isnan(up) & ~np.isnan(dn)
if sel.sum() > 1000:
    r_up = np.corrcoef(np.log1p(up[sel]), PM25[sel])[0, 1]
    r_dn = np.corrcoef(np.log1p(dn[sel]), PM25[sel])[0, 1]
    print(f"\ncorr(log1p(upwind FRP),   PM2.5) = {r_up:+.4f}")
    print(f"corr(log1p(downwind FRP), PM2.5) = {r_dn:+.4f}")
    if r_up > r_dn:
        print("Upwind fire tracks PM2.5 more strongly than downwind fire, which is "
              "the expected physical ordering - the bearing convention is right.")
    else:
        print("WARNING: downwind fire correlates at least as strongly as upwind. "
              "Check the wind-direction convention (met = direction wind comes FROM).")

Rows with a detection in the prior 24h, by radius:
     5 km :  4.58% of grid, max count 21, max FRP 144 MW
    10 km : 11.85% of grid, max count 60, max FRP 1,140 MW
    20 km : 24.88% of grid, max count 186, max FRP 1,558 MW
    50 km : 47.40% of grid, max count 761, max FRP 7,184 MW
   100 km : 64.76% of grid, max count 1464, max FRP 16,558 MW
   150 km : 72.78% of grid, max count 2250, max FRP 22,264 MW

fire_frp_sum_100km_24h
  mean PM2.5 when > 0 :   42.7 ug/m3  (n=1,102,218)
  mean PM2.5 when = 0 :   25.3 ug/m3  (n=603,034)

upwind_fire_frp_100km
  mean PM2.5 when > 0 :   44.9 ug/m3  (n=791,129)
  mean PM2.5 when = 0 :   29.3 ug/m3  (n=914,123)

crosswind_fire_frp_100km
  mean PM2.5 when > 0 :   44.2 ug/m3  (n=877,539)
  mean PM2.5 when = 0 :   28.5 ug/m3  (n=827,713)

downwind_fire_frp_100km
  mean PM2.5 when > 0 :   44.0 ug/m3  (n=818,412)
  mean PM2.5 when = 0 :   29.7 ug/m3  (n=886,840)

corr(log1p(upwind FRP),   PM2.5) = +0.2008
corr(log1p(downwind FRP), PM2.5) = +0.1734
Up

## 6. Regional and airshed context

PM2.5 is not purely local. A Delhi episode is a Punjab-to-Delhi transport event, and a
station that only sees its own history cannot anticipate one arriving.

Two guards apply throughout this section:

- **Leave-one-out.** A station never contributes to its own regional aggregate. Without
  this the "regional mean" partly *is* the target, which reads as a spectacular feature
  and transfers nowhere.
- **Direction.** The same sector machinery as fire: upwind neighbours are the ones the
  air is arriving from.

The network is sparse — mean nearest-neighbour distance is around 100 km — so the 25 km
ring will be empty for most stations. Coverage is reported rather than hidden, because a
mostly-NaN feature is worth knowing about before it reaches a model.

In [9]:
# ============================================================================
# REGIONAL CONTEXT - radius rings and upwind neighbours, leave-one-out
# ============================================================================
t0 = time.time()
lat_a = stations.lat.to_numpy("float64")
lon_a = stations.lon.to_numpy("float64")

# Pairwise station distances and bearings. 149x149 - trivially small.
D = np.zeros((S, S)); B = np.zeros((S, S))
for i in range(S):
    D[i] = haversine_km(lat_a[i], lon_a[i], lat_a, lon_a)
    B[i] = bearing_deg(lat_a[i], lon_a[i], lat_a, lon_a)
np.fill_diagonal(D, np.inf)                     # self can never be a neighbour
nb_sec = (np.floor(B / SECTOR_DEG).astype("int64")) % NSEC

for r in CONFIG["regional_radii_km"]:
    feat[f"regional_pm25_mean_{r}km"] = np.full((S, T), np.nan, dtype="float32")
feat["regional_pm25_max_50km"] = np.full((S, T), np.nan, dtype="float32")
feat["regional_pm25_std_50km"] = np.full((S, T), np.nan, dtype="float32")
feat["regional_station_count_50km"] = np.zeros((S, T), dtype="float32")
for r in [20, 50, 100]:
    feat[f"upwind_pm25_{r}km"] = np.full((S, T), np.nan, dtype="float32")
for n in ["upwind_pm25_mean", "upwind_pm25_max", "upwind_pm25_gradient"]:
    feat[n] = np.full((S, T), np.nan, dtype="float32")

ring_counts = {}
for i in range(S):
    for r in CONFIG["regional_radii_km"]:
        idx = np.where(D[i] <= r)[0]
        ring_counts.setdefault(r, []).append(len(idx))
        if len(idx) == 0:
            continue
        block = PM25[idx]                                    # (k, T)
        feat[f"regional_pm25_mean_{r}km"][i] = np.nanmean(block, axis=0)
        if r == 50:
            feat["regional_pm25_max_50km"][i] = np.nanmax(block, axis=0)
            feat["regional_pm25_std_50km"][i] = (np.nanstd(block, axis=0)
                                                 if len(idx) > 1 else np.nan)
            feat["regional_station_count_50km"][i] = np.sum(~np.isnan(block), axis=0)

    # Directional neighbours, all radii at once.
    for r in [20, 50, 100]:
        idx = np.where(D[i] <= r)[0]
        if len(idx) == 0:
            continue
        mask = M_UP[wind_sec[i]][:, nb_sec[i, idx]]          # (T, k)
        vals = PM25[idx].T                                    # (T, k)
        sel = np.where(mask > 0, vals, np.nan)
        with np.errstate(invalid="ignore"):
            feat[f"upwind_pm25_{r}km"][i] = np.where(
                wind_ok[i], np.nanmean(sel, axis=1), np.nan)

    idx = np.where(D[i] <= 150)[0]
    if len(idx):
        mask = M_UP[wind_sec[i]][:, nb_sec[i, idx]]
        sel = np.where(mask > 0, PM25[idx].T, np.nan)
        with np.errstate(invalid="ignore"):
            feat["upwind_pm25_mean"][i] = np.where(wind_ok[i], np.nanmean(sel, axis=1), np.nan)
            feat["upwind_pm25_max"][i] = np.where(wind_ok[i], np.nanmax(sel, axis=1), np.nan)

# Positive gradient = the air upwind is dirtier than here, i.e. something is on
# its way. This is the single most direct transport-warning feature available.
feat["upwind_pm25_gradient"] = (feat["upwind_pm25_mean"] - PM25).astype("float32")

# National leave-one-out context: is this a local event or a nationwide one?
tot = np.nansum(PM25, axis=0)
cnt = np.sum(~np.isnan(PM25), axis=0).astype("float64")
own = np.where(np.isnan(PM25), 0.0, PM25)
own_cnt = (~np.isnan(PM25)).astype("float64")
loo_n = np.maximum(cnt[None, :] - own_cnt, 1)
feat["national_pm25_mean"] = ((tot[None, :] - own) / loo_n).astype("float32")
feat["national_station_count"] = np.broadcast_to(cnt, (S, T)).astype("float32")
feat["pm25_local_excess"] = (PM25 - feat["national_pm25_mean"]).astype("float32")

print(f"Regional features in {time.time()-t0:.0f}s")
print("\nNeighbour counts per ring (stations within radius, excluding self):")
for r, counts in ring_counts.items():
    counts = np.array(counts)
    print(f"  {r:>4} km : median {np.median(counts):.0f}, "
          f"{(counts == 0).mean():.0%} of stations have none")
print("\nCoverage of the regional features:")
for n in [f"regional_pm25_mean_{r}km" for r in CONFIG["regional_radii_km"]] + \
         ["upwind_pm25_mean", "upwind_pm25_gradient"]:
    print(f"  {n:28} {(~np.isnan(feat[n]) & observed).sum() / observed.sum():6.1%} of observed rows")

# Leakage guard - a leave-one-out feature must never equal the target exactly.
ident = np.nanmean(np.round(feat["national_pm25_mean"], 6) == np.round(PM25, 6))
print(f"\nRows where national_pm25_mean == own pm25: {ident:.4%} (must be ~0)")

Regional features in 0s

Neighbour counts per ring (stations within radius, excluding self):
    25 km : median 1, 41% of stations have none
    50 km : median 2, 32% of stations have none
   100 km : median 2, 14% of stations have none

Coverage of the regional features:
  regional_pm25_mean_25km       56.9% of observed rows
  regional_pm25_mean_50km       66.0% of observed rows
  regional_pm25_mean_100km      84.4% of observed rows
  upwind_pm25_mean              63.7% of observed rows
  upwind_pm25_gradient          63.7% of observed rows

Rows where national_pm25_mean == own pm25: 0.0003% (must be ~0)


## 7. Atmospheric composition

CAMS gives coverage *between* stations, which 149 point monitors cannot. AOD is the
headline variable — column-integrated aerosol is the standard predictor in published
hyper-local PM2.5 work.

Two rules from the acquisition notebook are enforced in code here:

1. `cams_pm25` is **dropped**. It is a model's estimate of our target; a model trained on
   it would learn to copy CAMS, score well, and transfer nowhere.
2. Everything retained keeps the `cams_` prefix, so the whole model-derived block can be
   selected or removed in one operation — which is exactly what the A4 ablation does.

`satellite_age_hours` is 0.5 h throughout this configuration because CAMS is hourly and
complete. The column exists anyway so that a sparser satellite source can be substituted
later without a schema change, and so that no downstream code forward-fills a stale
observation without being able to see how stale it is.

In [10]:
# ============================================================================
# COMPOSITION FEATURES - CAMS, with change and anomaly
# ============================================================================
t0 = time.time()
# cams_pm25 is a model estimate of the target. Excluded by construction, not by
# a downstream filter that someone can forget to apply.
CAMS_KEEP = [c for c in CAMS_VARS if c != "cams_pm25"]
CAMS_EXCLUDED = [c for c in CAMS_VARS if c not in CAMS_KEEP]

SAT = gather_hourly(satellite, CAMS_KEEP, "satellite")
for name, mat in SAT.items():
    feat[name] = mat

for var in ["cams_aod", "cams_no2", "cams_co", "cams_dust"]:
    if var not in SAT:
        continue
    feat[f"{var}_change_3h"] = (SAT[var] - shift_time(SAT[var], 3)).astype("float32")
    feat[f"{var}_change_24h"] = (SAT[var] - shift_time(SAT[var], 24)).astype("float32")
    # Anomaly against this location's own trailing week: an AOD of 0.6 means
    # something different over the Thar than over Kerala.
    m, s = roll_mean_std(SAT[var], 168, shift=1)
    feat[f"{var}_anomaly"] = ((SAT[var] - m) /
                              np.where(s < 1e-6, np.nan, s)).astype("float32")

feat["cams_aod_regional_mean"] = np.full((S, T), np.nan, dtype="float32")
for i in range(S):
    idx = np.where(D[i] <= 100)[0]
    if len(idx):
        feat["cams_aod_regional_mean"][i] = np.nanmean(SAT["cams_aod"][idx], axis=0)

# Freshness columns. Constant here by construction; present so the contract does
# not change when a sparser source replaces CAMS.
feat["satellite_observation_available"] = (~np.isnan(SAT["cams_aod"])).astype("float32")
feat["satellite_age_hours"] = np.where(np.isnan(SAT["cams_aod"]), np.nan, 0.5).astype("float32")

print(f"Composition features in {time.time()-t0:.0f}s")
print(f"  retained : {CAMS_KEEP}")
print(f"  EXCLUDED : {CAMS_EXCLUDED}  (model estimate of the target)")
o = observed
for v in CAMS_KEEP:
    sel = o & ~np.isnan(feat[v])
    r = np.corrcoef(feat[v][sel], PM25[sel])[0, 1] if sel.sum() > 1000 else np.nan
    print(f"  corr({v:10}, PM2.5) = {r:+.4f}   coverage {sel.sum()/o.sum():.1%}")

Composition features in 0s
  retained : ['cams_pm10', 'cams_aod', 'cams_no2', 'cams_co', 'cams_so2', 'cams_dust', 'cams_o3']
  EXCLUDED : ['cams_pm25']  (model estimate of the target)
  corr(cams_pm10 , PM2.5) = +0.0865   coverage 100.0%
  corr(cams_aod  , PM2.5) = +0.0433   coverage 100.0%
  corr(cams_no2  , PM2.5) = +0.2722   coverage 100.0%
  corr(cams_co   , PM2.5) = +0.4188   coverage 100.0%
  corr(cams_so2  , PM2.5) = +0.2425   coverage 100.0%
  corr(cams_dust , PM2.5) = +0.0124   coverage 100.0%
  corr(cams_o3   , PM2.5) = +0.0566   coverage 100.0%


## 8. Terrain, calendar, and interactions

Terrain explains persistent differences between nearby stations. The calendar explains
the two things an annual sinusoid cannot: Diwali, which moves through the Gregorian
calendar, and the crop-burning windows.

The interaction terms encode compound conditions that a tree can only reach by splitting
several times — a shallow boundary layer *with* upwind fire is the recipe for a severe
episode, and giving the model the product lets one split express it.

In [11]:
# ============================================================================
# CALENDAR, TERRAIN AND INTERACTIONS
# ============================================================================
t0 = time.time()
# Local time, not UTC. India is UTC+05:30, so a UTC hour puts the morning peak
# at 03:30 and a UTC day-of-week mislabels every hour from 18:30 local onward.
local_hours = GRID_HOURS.tz_convert("Asia/Kolkata")
row_of = lambda v: np.broadcast_to(np.asarray(v, dtype="float32"), (S, T)).copy()

feat["hour"] = row_of(local_hours.hour)
feat["dow"] = row_of(local_hours.dayofweek)
feat["doy"] = row_of(local_hours.dayofyear)
feat["month"] = row_of(local_hours.month)
feat["is_weekend"] = row_of((local_hours.dayofweek >= 5).astype(int))
# Hour and day-of-year are circular: 23:00 and 00:00 are adjacent but 23 apart
# numerically, which trees split badly.
feat["sin_hour"] = row_of(np.sin(2 * np.pi * local_hours.hour / 24))
feat["cos_hour"] = row_of(np.cos(2 * np.pi * local_hours.hour / 24))
feat["sin_doy"] = row_of(np.sin(2 * np.pi * local_hours.dayofyear / 365.25))
feat["cos_doy"] = row_of(np.cos(2 * np.pi * local_hours.dayofyear / 365.25))

CAL_COLS = ["is_holiday", "diwali_window", "days_to_diwali", "days_after_diwali",
            "festival_window", "days_to_nearest_festival",
            "crop_burning_kharif", "crop_burning_rabi", "crop_burning_window"]
cal_idx = calendar.set_index("date")
local_dates = pd.DatetimeIndex(local_hours.tz_localize(None).normalize())
missing_dates = local_dates.difference(cal_idx.index)
if len(missing_dates):
    raise ValueError(f"{len(missing_dates)} grid dates absent from the calendar table "
                     f"(first {missing_dates[0].date()}). Extend it in notebook 05.")
for c in CAL_COLS:
    feat[c] = row_of(cal_idx.loc[local_dates, c].to_numpy())

GEO_COLS = ["elevation_m", "elevation_std_10km", "elevation_range_10km",
            "elevation_relief_10km", "is_indo_gangetic_plain"]
geo = (stations[["lat", "lon"]].merge(terrain, on=["lat", "lon"], how="left"))
if geo[GEO_COLS].isna().any().any():
    raise ValueError("terrain layer does not cover every station")
col_of = lambda v: np.broadcast_to(np.asarray(v, dtype="float32")[:, None], (S, T)).copy()
for c in GEO_COLS:
    feat[c] = col_of(geo[c].to_numpy())
feat["lat"] = col_of(stations.lat.to_numpy())
feat["lon"] = col_of(stations.lon.to_numpy())

# Interactions. Each is a compound physical condition, not an arbitrary product.
feat["low_pbl_x_high_pm25"] = (feat["low_pbl_flag"] * PM25).astype("float32")
feat["low_wind_x_high_pm25"] = (feat["low_wind_flag"] * PM25).astype("float32")
feat["humidity_x_pm25"] = (WX["relative_humidity_2m"] * PM25 / 100.0).astype("float32")
# Rain scavenges smoke, so fire evidence must be discounted when it is raining.
feat["rain_x_fire"] = (feat["rain_24h"] *
                       np.log1p(feat["fire_frp_sum_100km_24h"])).astype("float32")
# Trapped smoke: a shallow layer over active upwind fire is the severe-episode
# recipe, and neither term alone identifies it.
feat["low_pbl_x_fire"] = (feat["low_pbl_flag"] *
                          np.log1p(feat["upwind_fire_frp_100km"])).astype("float32")
feat["stagnation_x_upwind_pm25"] = (feat["stagnation_flag"] *
                                    feat["upwind_pm25_mean"]).astype("float32")

print(f"Calendar / terrain / interactions in {time.time()-t0:.0f}s")
print(f"Total feature matrices: {len(feat)}")

# Local-time sanity check: reshape the grid into whole local days and ask when
# the daily maximum occurs. A UTC-derived hour column would put the morning
# traffic peak at 03:30, so this is the check that the timezone handling holds.
first_midnight = int(np.argmax(local_hours.hour == 0))
n_days = (T - first_midnight) // 24
day_block = PM25[:, first_midnight:first_midnight + n_days * 24].reshape(S, n_days, 24)
with np.errstate(invalid="ignore"):
    has_day = np.sum(~np.isnan(day_block), axis=2) >= 12
    argmax_h = np.where(has_day, np.nanargmax(np.nan_to_num(day_block, nan=-1), axis=2), -1)
peak_hours = pd.Series(argmax_h[argmax_h >= 0]).value_counts().sort_index()
print("\nLocal hour of the daily PM2.5 maximum (days with >=12 observations):")
print(peak_hours.to_string())
del day_block

Calendar / terrain / interactions in 0s
Total feature matrices: 183

Local hour of the daily PM2.5 maximum (days with >=12 observations):
0     6339
1     2591
2     1850
3     1525
4     1278
5     1308
6     1698
7     3382
8     5282
9     4489
10    3021
11    2468
12    2297
13    2202
14    2070
15    1757
16    1686
17    1719
18    2263
19    3884
20    5473
21    5615
22    4575
23    6127


## 9. Pollution events

An event is a contiguous run of hours above the threshold. Phase 7 scores alert lead time
against these, so their boundaries must be honest.

Working on the dense grid fixes a subtle v2 defect for free: a data gap now **ends** an
event, because gap hours are NaN and therefore not above threshold. In v2 an event could
silently bridge a gap of arbitrary length and report a duration that never happened.

In [12]:
# ============================================================================
# POLLUTION EVENTS - contiguous episodes per station
# ============================================================================
THRESH = CONFIG["event_threshold"]
polluted = np.nan_to_num(PM25, nan=-1.0) >= THRESH        # NaN never starts an event
prev = np.concatenate([np.zeros((S, 1), bool), polluted[:, :-1]], axis=1)
start_flag = polluted & ~prev
event_no = np.cumsum(start_flag, axis=1)
event_id = np.where(polluted, event_no, np.nan)

feat["event_start_flag"] = start_flag.astype("float32")

n_events = int(start_flag.sum())
si, ti = np.nonzero(polluted)
ev_frame = pd.DataFrame({
    "station": si, "t": ti,
    "event_no": event_no[si, ti], "pm25": PM25[si, ti],
})
ev = (ev_frame.groupby(["station", "event_no"])
              .agg(t_start=("t", "min"), t_end=("t", "max"),
                   event_peak_pm25=("pm25", "max"),
                   event_mean_pm25=("pm25", "mean"),
                   event_observed_hours=("t", "size"))
              .reset_index())
ev["event_duration_hours"] = ev.t_end - ev.t_start + 1
ev["event_start"] = GRID_HOURS[ev.t_start.to_numpy()]
ev["event_end"] = GRID_HOURS[ev.t_end.to_numpy()]

print(f"Events (>= {THRESH:.0f} ug/m3) : {n_events:,}")
print(f"  duration hours : median {ev.event_duration_hours.median():.0f}, "
      f"max {ev.event_duration_hours.max():.0f}")
print(f"  peak PM2.5     : median {ev.event_peak_pm25.median():.0f}, "
      f"max {ev.event_peak_pm25.max():.0f}")
print(f"  extreme events (peak >= 150) : {(ev.event_peak_pm25 >= 150).sum():,}")
# On the dense grid a gap terminates an event, so every event is contiguous by
# construction. Asserting it makes a future change to `polluted` visible.
assert (ev.event_duration_hours == ev.event_observed_hours).all(), \
    "an event spans a gap - contiguity assumption broken"
print("  all events contiguous (a data gap ends an event, as it should)")

Events (>= 60 ug/m3) : 44,563
  duration hours : median 3, max 354
  peak PM2.5     : median 79, max 1870
  extreme events (peak >= 150) : 4,952
  all events contiguous (a data gap ends an event, as it should)


## 10. Targets

Three families, because the product has three different questions and one regression
cannot answer all of them.

- **Concentration** at `t+h` — the forecast curve.
- **Peak** over the next `h` hours — closer to the real alerting question than any
  single future hour, since an episode that peaks at 03:00 matters whether or not
  03:00 is the hour being scored.
- **Hazard** — will *any* of the next 24 hours breach the threshold? Phase 5 showed
  regression-thresholding collapses to exactly zero extreme recall at 24 h, so the
  alert must be a classification target in its own right.

Because targets are built on the dense grid, `t+24` means 24 hours ahead. In v2 it meant
24 *observations* ahead, which for a gappy station was a different and unstated horizon.

In [13]:
# ============================================================================
# FORECAST TARGETS - negative shifts only
# ============================================================================
targets = {}
for h in CONFIG["horizons"]:
    targets[f"target_pm25_t_plus_{h}h"] = shift_time(PM25, -h)

# Peak over the next w hours: max of the w forward shifts. Built by reversing
# the array so the same causal window helper can be reused.
rev = PM25[:, ::-1]
for w in CONFIG["peak_windows"]:
    # Window of width w ending at (reversed) t-1 == hours t+1..t+w forward.
    win = roll_windows(rev, w, shift=1)
    targets[f"target_peak_pm25_next_{w}h"] = np.nanmax(win, axis=2)[:, ::-1].astype("float32")
    del win
targets["target_max_pm25_next_24h"] = targets["target_peak_pm25_next_24h"]

peak24 = targets["target_peak_pm25_next_24h"]
targets["target_high_within_24h"] = np.where(np.isnan(peak24), np.nan, peak24 >= 60).astype("float32")
targets["target_very_high_within_24h"] = np.where(np.isnan(peak24), np.nan, peak24 >= 90).astype("float32")
targets["target_extreme_within_24h"] = np.where(np.isnan(peak24), np.nan, peak24 >= 150).astype("float32")

print(f"Built {len(targets)} targets\n")
print("Availability on observed rows (later horizons lose the tail of each series):")
for name in targets:
    v = targets[name]
    print(f"  {name:32} {(~np.isnan(v) & observed).sum():>9,} "
          f"({(~np.isnan(v) & observed).sum()/observed.sum():6.1%})")
print("\nAlerting class balance:")
for name in ["target_high_within_24h", "target_very_high_within_24h",
             "target_extreme_within_24h"]:
    v = targets[name][observed]
    v = v[~np.isnan(v)]
    print(f"  {name:32} {int(v.sum()):>8,} positives ({v.mean():.2%})")
if np.nansum(targets["target_extreme_within_24h"][observed]) == 0:
    print("\nWARNING: no extreme events anywhere - Phase 7 alerting cannot be trained.")

Built 14 targets

Availability on observed rows (later horizons lose the tail of each series):
  target_pm25_t_plus_1h            1,648,494 ( 96.7%)
  target_pm25_t_plus_3h            1,611,870 ( 94.5%)
  target_pm25_t_plus_6h            1,584,425 ( 92.9%)
  target_pm25_t_plus_12h           1,552,777 ( 91.1%)
  target_pm25_t_plus_24h           1,534,017 ( 90.0%)
  target_pm25_t_plus_48h           1,506,929 ( 88.4%)
  target_peak_pm25_next_6h         1,694,785 ( 99.4%)
  target_peak_pm25_next_12h        1,699,657 ( 99.7%)
  target_peak_pm25_next_24h        1,703,119 ( 99.9%)
  target_peak_pm25_next_48h        1,704,218 ( 99.9%)
  target_max_pm25_next_24h         1,703,119 ( 99.9%)
  target_high_within_24h           1,703,119 ( 99.9%)
  target_very_high_within_24h      1,703,119 ( 99.9%)
  target_extreme_within_24h        1,703,119 ( 99.9%)

Alerting class balance:
  target_high_within_24h            639,435 positives (37.54%)
  target_very_high_within_24h       333,869 positives (19.60%

## 11. Assemble, audit, persist

The audit is the point of this section. Every claim made above about causality is
checked against the assembled frame rather than trusted, because a leakage bug is
invisible in the metrics that would otherwise catch it — it makes them *better*.

In [14]:
# ============================================================================
# ASSEMBLE - matrices to frame, then drop the padding rows
# ============================================================================
t0 = time.time()
# The padding rows exist so the windows above are correctly spaced. They are not
# observations and must never become training rows. Masking DURING assembly
# rather than after keeps peak memory near the size of the result instead of
# building a 2-million-row frame and then copying 1.7 million of it.
keep = observed.ravel()

frame = pd.DataFrame({
    "location_id": np.repeat(stations.location_id.to_numpy(), T)[keep],
    "timestamp_utc": np.tile(naive64(GRID_HOURS), S)[keep],
})
for c in ["sensor_id", "station_name", "source", "source_quality"]:
    frame[c] = np.repeat(stations[c].to_numpy(), T)[keep]

# pop() as we go, so the feature dict shrinks while the frame grows.
for name in list(feat):
    mat = feat.pop(name)
    if mat.shape != (S, T):
        raise ValueError(f"feature {name} has shape {mat.shape}, expected {(S, T)}")
    frame[name] = mat.ravel()[keep]
for name in list(targets):
    frame[name] = targets.pop(name).ravel()[keep]
frame["event_id"] = event_id.ravel()[keep]

frame["timestamp_utc"] = pd.to_datetime(frame["timestamp_utc"], utc=True)
frame["timestamp_local"] = frame["timestamp_utc"].dt.tz_convert("Asia/Kolkata")
month = frame["month"].to_numpy()
frame["season"] = np.select(
    [np.isin(month, [10, 11]), np.isin(month, [12, 1, 2]), np.isin(month, [3, 4, 5])],
    ["post_monsoon", "winter", "pre_monsoon"], default="monsoon")
frame["pollution_regime"] = pd.cut(frame["pm25"], CONFIG["regime_bins"],
                                   labels=CONFIG["regime_labels"])

for c in frame.columns:
    if frame[c].dtype == "float64":
        frame[c] = frame[c].astype("float32")

print(f"Assembled in {time.time()-t0:.0f}s")
print(f"Rows      : {len(frame):,}  (grid {S*T:,} minus {(~observed).sum():,} padding)")
print(f"Columns   : {frame.shape[1]}")
print(f"Memory    : {frame.memory_usage(deep=True).sum()/1e9:.2f} GB")
print(f"\nSeasons:\n{frame.season.value_counts().to_string()}")
print(f"\nRegimes:\n{frame.pollution_regime.value_counts().sort_index().to_string()}")

Assembled in 1s
Rows      : 1,705,252  (grid 2,029,380 minus 324,128 padding)
Columns   : 208
Memory    : 1.58 GB

Seasons:
season
monsoon         673073
pre_monsoon     562191
winter          280215
post_monsoon    189773

Regimes:
pollution_regime
normal       912999
moderate     540150
high         163172
very_high     70258
extreme       18673


In [15]:
# ============================================================================
# LEAKAGE AUDIT - assert causality rather than assuming it
# ============================================================================
# Each check corresponds to a way this pipeline could silently learn the answer.
# A leak does not show up as an error; it shows up as a better score, which is
# why it has to be tested for directly.
# ============================================================================
audit, problems = {}, []


def check(name, ok, detail=""):
    audit[name] = {"pass": bool(ok), "detail": detail}
    print(f"  [{'PASS' if ok else 'FAIL'}] {name:44} {detail}")
    if not ok:
        problems.append(name)


TARGET_COLS = [c for c in frame.columns if c.startswith("target_")]
ID_COLS = ["location_id", "sensor_id", "station_name", "source", "source_quality",
           "timestamp_utc", "timestamp_local", "season", "pollution_regime", "event_id"]
FEATURE_COLS = [c for c in frame.columns
                if c not in ID_COLS and c not in TARGET_COLS
                and pd.api.types.is_numeric_dtype(frame[c])]

print("Causality and contract audit:")
check("no target column in the feature list",
      not any(c.startswith("target_") for c in FEATURE_COLS),
      f"{len(FEATURE_COLS)} features, {len(TARGET_COLS)} targets")
check("cams_pm25 excluded", "cams_pm25" not in frame.columns,
      "model estimate of the target")
check("no duplicate station-hours",
      not frame.duplicated(subset=["location_id", "timestamp_utc"]).any())
check("every row is an observation", frame.pm25.notna().all(),
      f"{frame.pm25.isna().sum()} NaN targets")
check("no negative PM2.5", (frame.pm25 >= 0).all(),
      f"min {frame.pm25.min():.1f}")

# The strongest single test: correlate every feature with the 1h target. A
# feature that is near-perfectly correlated is the answer wearing a disguise.
# Sample first, then widen - selecting 250 columns across 1.7M rows before
# sampling would copy the whole frame.
probe = frame.sample(min(200_000, len(frame)), random_state=0)
probe = probe.dropna(subset=["target_pm25_t_plus_1h"])
corr = probe[FEATURE_COLS].corrwith(probe["target_pm25_t_plus_1h"]).abs().sort_values(ascending=False)
suspicious = corr[corr > 0.999]
check("no feature is a copy of the target", len(suspicious) == 0,
      f"max |corr| {corr.iloc[0]:.4f} ({corr.index[0]})")
print("\n  Top feature/target correlations (high is expected for lags; 1.000 is not):")
print(corr.head(8).to_string().replace("\n", "\n    "))

# A forward-looking window would make the peak target reachable from a feature.
peak_corr = frame[["pm25_roll_max_24h", "target_peak_pm25_next_24h"]].dropna()
check("trailing max is not the forward peak",
      peak_corr.corr().iloc[0, 1] < 0.99,
      f"corr {peak_corr.corr().iloc[0, 1]:.4f}")

# Event aggregates describe an episode INCLUDING its future, so they must not be
# features. Only the causal start flag and the id survive.
check("event aggregates are not features",
      not any(c.startswith("event_") and c not in ("event_start_flag",)
              for c in FEATURE_COLS),
      "only event_start_flag is retained")

nulls = frame[FEATURE_COLS].isna().mean().sort_values(ascending=False)
print(f"\n  Features more than 50% null: {(nulls > 0.5).sum()}")
if (nulls > 0.5).any():
    print(nulls[nulls > 0.5].head(10).to_string().replace("\n", "\n    "))

if problems:
    raise RuntimeError(f"Leakage audit failed: {problems}")
print("\nAudit passed.")

Causality and contract audit:
  [PASS] no target column in the feature list         184 features, 14 targets
  [PASS] cams_pm25 excluded                           model estimate of the target
  [PASS] no duplicate station-hours                   
  [PASS] every row is an observation                  0 NaN targets
  [PASS] no negative PM2.5                            min 0.0


  [PASS] no feature is a copy of the target           max |corr| 0.8784 (pm25)

  Top feature/target correlations (high is expected for lags; 1.000 is not):
pm25                   0.878423
    humidity_x_pm25        0.765002
    pm25_lag_1h            0.762884
    low_pbl_x_high_pm25    0.739773
    pm25_roll_mean_3h      0.738500
    pm25_local_excess      0.726280
    pm25_roll_mean_24h     0.706613
    pm25_roll_mean_6h      0.698385
  [PASS] trailing max is not the forward peak         corr 0.4934
  [PASS] event aggregates are not features            only event_start_flag is retained

  Features more than 50% null: 8
fire_min_distance_5km_24h     0.954709
    fire_min_distance_10km_24h    0.882331
    fire_min_distance_20km_24h    0.752674
    upwind_pm25_20km              0.720187
    upwind_pm25_50km              0.657715
    fire_min_distance_50km_24h    0.528229
    upwind_pm25_100km             0.507697
    regional_pm25_std_50km        0.500232

Audit passed.


In [16]:
# ============================================================================
# PERSIST the event-aware feature table + manifest
# ============================================================================
EVENT_FILE = EVENT_DIR / "pm25_event_aware_features.parquet"
frame.to_parquet(EVENT_FILE, index=False)


def _fingerprint(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()[:16]


# Each feature belongs to EXACTLY ONE group. The A0-A6 ablation adds groups
# cumulatively, so a feature counted twice would make one ablation step appear
# to contribute information that a previous step had already supplied.
CALENDAR_BASE = {"hour", "dow", "doy", "month", "is_weekend",
                 "sin_hour", "cos_hour", "sin_doy", "cos_doy"}
STATIC_COLS = set(GEO_COLS) | {"lat", "lon"}


def _group_of(c):
    if "_x_" in c:
        return "interaction"                      # tested first: it spans sources
    if "fire" in c:
        return "fire"
    if c.startswith(("cams_", "satellite_")):
        return "satellite"
    if c.startswith(("regional_", "upwind_pm25", "national_")) or c == "pm25_local_excess":
        return "regional"
    if c in STATIC_COLS:
        return "static"
    if c in CAL_COLS or c in CALENDAR_BASE:
        return "calendar"
    if c in WEATHER_VARS or c.startswith(("rain_", "delta_", "ventilation",
                                          "stagnation", "low_wind", "low_pbl")):
        return "weather"
    # event_start_flag is pm25[t] crossing the threshold, so it is known at t
    # and belongs with the target's own history.
    if c.startswith("pm25") or c == "event_start_flag":
        return "temporal"
    return "ungrouped"


FEATURE_GROUPS = {}
for c in FEATURE_COLS:
    FEATURE_GROUPS.setdefault(_group_of(c), []).append(c)
ungrouped = FEATURE_GROUPS.pop("ungrouped", [])
assert sum(len(v) for v in FEATURE_GROUPS.values()) + len(ungrouped) == len(FEATURE_COLS)

manifest = {
    "dataset_version": CONFIG["dataset_version"],
    "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "source": "phase1_canonical_dataset + phase7_driver_layers",
    "rows": int(len(frame)),
    "stations": int(frame.location_id.nunique()),
    "time_span": [str(frame.timestamp_utc.min()), str(frame.timestamp_utc.max())],
    "grid": {"stations": S, "hours": T, "cells": S * T,
             "observed_fraction": float(observed.mean())},
    "seasons": {k: int(v) for k, v in frame.season.value_counts().items()},
    "extreme_rows": int((frame.pm25 >= 150).sum()),
    "severe_rows": int((frame.pm25 >= 90).sum()),
    "event_count": n_events,
    "n_features": len(FEATURE_COLS),
    "features": FEATURE_COLS,
    "feature_groups": {k: sorted(v) for k, v in FEATURE_GROUPS.items()},
    "ungrouped_features": ungrouped,
    "targets": TARGET_COLS,
    "config": {k: v for k, v in CONFIG.items() if k != "regime_bins"},
    "leakage_audit": audit,
    "design_notes": [
        "All windows are computed on a dense hourly grid, so a 24h window is 24 "
        "hours rather than 24 observations. v2 counted rows and a station with a "
        "7320-hour gap therefore had a '24h' mean spanning most of a year.",
        "Fire bearings are discretised into 16 sectors of 22.5 deg; the +-45 deg "
        "upwind cone is resolved to that granularity.",
        "cams_* are CAMS reanalysis model output, not observations. Predictor "
        "only; cams_pm25 is excluded entirely.",
        "satellite_age_hours is 0.5h throughout because CAMS is hourly and "
        "complete; the column exists for contract stability with sparser sources.",
    ],
    "known_gaps": [
        "weather features are observations at t, not forecasts valid at t+h "
        "(roadmap section 30) - causally safe but weaker at long horizons",
        "population/road/building density and land use not acquired (roadmap "
        "section 15) - needs WorldPop/GHSL/OSM",
        "the 25km regional ring is empty for most stations; the network is sparse",
    ],
    "sha256_16": _fingerprint(EVENT_FILE),
}
with open(EVENT_DIR / "dataset_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2, default=str)

print("PHASE 2 COMPLETE\n")
print(f"  version   : {manifest['dataset_version']}")
print(f"  rows      : {manifest['rows']:,}")
print(f"  features  : {manifest['n_features']}  (v2 had 96)")
print(f"  targets   : {len(TARGET_COLS)}  (v2 had 9)")
print(f"  events    : {manifest['event_count']:,}")
print(f"  extreme   : {manifest['extreme_rows']:,} rows")
print(f"  fingerprint: {manifest['sha256_16']}")
print("\n  Feature groups (the A0-A6 ablation consumes these):")
for k, v in manifest["feature_groups"].items():
    print(f"    {k:12} {len(v):>3}")
if ungrouped:
    print(f"    ungrouped   {len(ungrouped):>3}  {ungrouped}")
print(f"\nWrote: {EVENT_FILE}  ({EVENT_FILE.stat().st_size/1e6:.0f} MB)")
print("Notebooks 06, 07 and 08 read this file.")

PHASE 2 COMPLETE

  version   : 3.0.0
  rows      : 1,705,252
  features  : 184  (v2 had 96)
  targets   : 14  (v2 had 9)
  events    : 44,563
  extreme   : 19,057 rows
  fingerprint: ba72d91a76c075ee

  Feature groups (the A0-A6 ablation consumes these):
    temporal      42
    weather       28
    fire          46
    regional      15
    satellite     22
    calendar      18
    static         7
    interaction    6

Wrote: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator/data/pm25/processed/event_aware/pm25_event_aware_features.parquet  (433 MB)
Notebooks 06, 07 and 08 read this file.
